# Pool probe — do we have the questions?

One question decides the whole data strategy: does the public pool store the **prompt text** each
passage answers, or only an opaque id? If the text is there, we can regenerate the missing models on
the host's own prompts. If it is not, no generation can be matched to the host.

In [ ]:
"""Does the public pool carry the prompt text?

Everything about generating matched data depends on this one answer. If the pool stores the question
each passage answers, we can regenerate the missing models on the host's own prompts. If it only
stores an opaque prompt_id, then no amount of generation can be matched to the host, and the only
paired data we will ever have is what is already published.
"""
import glob
import json

import pandas as pd


def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None


path = find("pool_v4.parquet") or find("*.parquet")
print("pool file:", path, flush=True)
df = pd.read_parquet(path)
print("shape:", df.shape, flush=True)
print("columns:", list(df.columns), flush=True)
print("dtypes:\n", df.dtypes, flush=True)

for c in df.columns:
    if df[c].dtype == object:
        vals = df[c].dropna().astype(str)
        if len(vals) == 0:
            continue
        print(f"\n--- column {c}: {vals.nunique():,} unique")
        for v in vals.drop_duplicates().head(3):
            print("   ", v[:300].replace("\n", " "))

# is any text column long enough to be a question rather than an answer?
for c in df.columns:
    if df[c].dtype == object:
        w = df[c].dropna().astype(str).str.split().str.len()
        if len(w):
            print(f"\n{c}: median words {w.median():.0f}, max {w.max()}")

# how many models per prompt, i.e. how paired is the pool really
if "prompt_id" in df.columns:
    per = df.groupby("prompt_id")["model"].nunique()
    print("\nprompts:", len(per), "| models per prompt: median", per.median(),
          "max", per.max(), "| full 18:", int((per == 18).sum()))
    print("rows per model:\n", df["model"].value_counts().to_string())

summary = {"columns": list(df.columns), "shape": list(df.shape)}
json.dump(summary, open("pool_probe.json", "w"), indent=2)
